In [0]:
%run ../00_common/data_utils

In [0]:
def generate_task_table(task_id, topic_batch_log_df, topic_priority_df, max_process_count: int = 100000):
    print(f"INFO - ===== 开始生成Task表 [Task_id: {task_id}] =====\n")
    print(f"INFO - 单次任务最大处理量：{max_process_count}")

    # 1. 筛选未处理数据 + 统一is_handle类型
    topic_batch_log_df = topic_batch_log_df.withColumn("is_handle", F.col("is_handle").cast("boolean"))
    unhandled_df = topic_batch_log_df.filter(F.col("is_handle") == False)
    unhandled_count = unhandled_df.count()
    print(f"INFO - 未处理的Batch总数：{unhandled_count}")

    # 定义Task表Schema
    task_schema = StructType([
        StructField("Task_id", StringType(), nullable=False),
        StructField("Batch_id_list", StringType(), nullable=True),
        StructField("Task_process_count", LongType(), nullable=False),
        StructField("Topic_Process_Count", StringType(), nullable=False),
        StructField("Max_Task_Process_count", LongType(), nullable=False),
        StructField("Create_Time", TimestampType(), nullable=False)
    ])
    
    if unhandled_count == 0:
        print(f"WARN - 无未处理的Batch数据，返回空Task表")
        return spark.createDataFrame([], schema=task_schema), topic_batch_log_df

    # 2. 关联优先级 + 按规则排序
    log_with_priority = unhandled_df.join(
        topic_priority_df.filter(F.col("Is_Active") == 1),
        on=unhandled_df["topic"] == topic_priority_df["Topic_Name"],
        how="inner"
    ).orderBy(
        F.col("Priority").desc(),    # 先按Topic优先级降序
        F.col("creation_time").asc(),# 同Topic按创建时间升序
        F.col("batch_number").asc()  # 同时间按批次号升序（兜底）
    )

    # 3. 按Topic分组，预存Batch和取数规则（核心）
    topic_batch_map = {}

    # 先按优先级排序Topic
    sorted_topics = log_with_priority.select("topic", "Priority", "Topic_Max_Process_Count").distinct().orderBy(F.col("Priority").desc()).collect()
    for row in sorted_topics:
        topic = row["topic"]
        priority = row["Priority"]
        topic_max_process_count = row["Topic_Max_Process_Count"]

        # 每个Topic只保留未处理、按时间排序的Batch
        topic_batches = log_with_priority.filter(F.col("topic") == topic) \
            .orderBy("creation_time", "batch_number") \
            .select("batch_number", "read_record_count") \
            .collect()

        topic_batch_map[topic] = {
            "batches": topic_batches,  # 该Topic的Batch列表（按时间排序）
            "max_batch_per_round": priority,  # 每轮最多取Priority个
            "topic_max_process_count": topic_max_process_count, # topic最多处理条数
            "ptr": 0  # 读取指针（记录取到第几个Batch，避免重复取）
        }

    # 4. 核心逻辑：按Topic轮询，每轮取Priority个Batch
    handled_batch_numbers = [] # 存储本次分配的Batch列表
    task_process_count = 0     # 累计处理数量（用于配额控制）
    topic_process_count_dict = {k: 0 for k in topic_batch_map}  # 存储每个topic处理条数
    create_time = datetime.now()

    print("INFO - 开始按Topic优先级轮询分配Batch...")
    while True:
        any_batch_allocated = False  # 只有真正分配了Batch，才认为本轮有进展
        
        # 按优先级遍历每个Topic（如：History → Aus → Hkg → Jpn）
        for topic in list(topic_batch_map.keys()):
            
            topic_info = topic_batch_map[topic]
            batches = topic_info["batches"]
            max_take = topic_info["max_batch_per_round"]
            ptr = topic_info["ptr"]

            topic_max_process_count = topic_info["topic_max_process_count"]
            topic_process_count = topic_process_count_dict[topic]

             # 配额已用完，终止整个循环
            if task_process_count >= max_process_count:
                break
            
            # 该Topic无剩余Batch 或者该Topic已达条数上限，跳过本次循环，进入下一循环
            if ptr >= len(batches) or topic_process_count >= topic_max_process_count:
                continue  
            
            # 本轮最多取：优先级数量 OR 剩余Batch数量（取最小值）
            take_num = min(max_take, len(batches) - ptr)

            # 遍历该Topic本轮可分配的Batch（如 His取2个，Aus 取 1 个）
            for i in range(take_num):
                batch_idx = ptr + i
                batch = batches[batch_idx]
                batch_number = batch["batch_number"]
                read_count = batch["read_record_count"]
                

                # topic配额检查：加入当前Batch会超topic配额，跳过
                if topic_process_count_dict[topic] + read_count > topic_max_process_count:
                    print(f"WARN - Topic [{topic}] Batch [{batch_number}] 超出Topic配额，跳过（累计: {topic_process_count_dict[topic]} + {read_count} > {topic_max_process_count}）")

                    topic_info["ptr"] += i
                    break  # 停止当前Topic的本轮分配

                # 总配额检查：加入当前Batch会超配额，跳过
                if task_process_count + read_count > max_process_count:
                    print(f"WARN - Topic [{topic}] Batch [{batch_number}] 超出总配额，跳过（累计: {task_process_count} + {read_count} > {max_process_count}）")
                    # 超配额时，指针移到当前Batch位置，下一轮不再检查这个Batch
                    topic_info["ptr"] += i
                    break  # 停止当前Topic的本轮分配
                
                # ✅ 真正分配了Batch，才标记为有进展
                handled_batch_numbers.append(batch_number)

                topic_process_count_dict[topic] = topic_process_count_dict[topic] + read_count
                task_process_count += read_count

                print(f"INFO - Topic [{topic}] 分配Batch [{batch_number}]({read_count}条), Topic条数累计: {topic_process_count_dict[topic]}, 总条数累计：{task_process_count}")
                topic_info["ptr"] += 1  # 指针+1：标记该Batch已处理，下次从下一个开始
                any_batch_allocated = True  # 标记本轮有实际分配

        # 🔴 核心退出条件：
        # 1. 本轮没有分配任何Batch → 所有Batch要么已处理，要么都超配额 → 退出
        # 2. 已达配额上限 → 退出
        if not any_batch_allocated:
            print("INFO - 无更多可分配Batch（或所有剩余Batch均超配额），停止分配")
            break
        if task_process_count >= max_process_count:
            print(f"INFO - 已达配额上限（{max_process_count}），停止分配")
            break

    # 5. 生成Task表
    print(f"\nINFO - 最终分配完成：共分配 {len(handled_batch_numbers)} 个Batch，累计处理 {task_process_count} 条记录")
    task_data = [{
        "Task_id": task_id,
        "Batch_id_list": ",".join(handled_batch_numbers) if handled_batch_numbers else "",
        "Task_process_count": task_process_count,
        "Topic_Process_Count": json.dumps(topic_process_count_dict),
        "Max_Task_Process_count": max_process_count,
        "Create_Time": create_time
    }]
    task_df = spark.createDataFrame(task_data, schema=task_schema)
    print(f"INFO - 已生成Task表 [Task_id: {task_id}]")

    # 6. 更新is_handle字段
    if handled_batch_numbers:
        updated_topic_batch_log_df = topic_batch_log_df.withColumn(
            "is_handle",
            F.when(F.col("batch_number").isin(handled_batch_numbers), True).otherwise(F.col("is_handle"))
        )
        current_updated_count = len(handled_batch_numbers)
        print(f"INFO - 本次更新 {current_updated_count} 个Batch的is_handle字段为TRUE")
    else:
        updated_topic_batch_log_df = topic_batch_log_df

    print(f"\nINFO - ===== Task表生成完成 =====\n")
    return task_df, updated_topic_batch_log_df

In [0]:
def main_process(task_id, max_process_count):
    # 读取batch_log表、priority表
    t_topic_batch_log = f"{get_env_config('config_database')}.t_topic_batch_log"
    t_topic_priority = f"{get_env_config('config_database')}.t_topic_priority"
    topic_batch_log_df = spark.table(t_topic_batch_log)
    topic_priority_df = spark.table(t_topic_priority)

    # 调用函数：为每个task_id分配batch_id_list
    task_df, updated_topic_batch_log_df = generate_task_table(task_id, topic_batch_log_df, topic_priority_df, int(max_process_count))

    # append到task log表
    t_task_batchlist_log = f"{get_env_config('config_database')}.t_task_batchlist_log"
    print(f"Append {task_df.count()} task records into {t_task_batchlist_log}")
    save_to_target_table(task_df,t_task_batchlist_log,f"task_id='{task_id}'")

    # batch_log表中处理过的batch改为true
    print(f"Updated {updated_topic_batch_log_df.count()} batch records into {t_topic_batch_log}")
    merge_condition = get_merge_condition(["batch_number"])
    merge_t_table(t_topic_batch_log, updated_topic_batch_log_df, merge_condition)

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

max_process_count = dbutils.widgets.get("max_process_count")
print(f"max_process_count: {max_process_count}")

with StepLogger("generate_task_table", "01-2", "consumerlist", task_id=task_id) as logger:
    main_process(task_id, max_process_count)